In [ ]:

from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.agents import create_agent
from langchain_ollama import ChatOllama


# =====================================================
# 1. LLM
# =====================================================

llm = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434"
)


# =====================================================
# 2. CREATE AGENTS
# =====================================================

research_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a cybersecurity research specialist."
)

validation_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You validate cybersecurity research. "
        "Return APPROVED if sufficient, otherwise REVISE."
    )
)

report_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You write cybersecurity reports."
)


# =====================================================
# 3. SHARED STATE
# =====================================================

class State(TypedDict):
    topic: str
    research: str
    validation: str
    report: str
    next_agent: str


# =====================================================
# 4. ROUTER NODE
# =====================================================

# the return means we are setting value in state we can also use modal.copy function
def router_node(state: State):

    print("\nRouter Node Executed")

    if not state["research"]:
        return {"next_agent": "research_agent"}

    elif not state["validation"]:
        return {"next_agent": "validation_agent"}

    elif state["validation"] == "APPROVED":
        return {"next_agent": "report_agent"}

    else:
        return {"next_agent": "research_agent"}


# =====================================================
# 5. RESEARCH NODE
# =====================================================

def research_node(state: State):

    print("\nResearch Agent Executed")

    result = research_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"Research this topic: {state['topic']}"
            }
        ]
    })

    research = result["messages"][-1].content

    return {
        "research": research,
        "validation": ""
    }


# =====================================================
# 6. VALIDATION NODE
# =====================================================

def validation_node(state: State):

    print("\nValidation Agent Executed")

    result = validation_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"Validate this research: {state['research']}"
            }
        ]
    })

    verdict = result["messages"][-1].content.strip()

    return {"validation": verdict}


# =====================================================
# 7. REPORT NODE
# =====================================================

def report_node(state: State):

    print("\nReport Agent Executed")

    result = report_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    f"Write a report about {state['topic']} "
                    f"using this research: {state['research']}"
                )
            }
        ]
    })

    return {
        "report": result["messages"][-1].content
    }


# =====================================================
# 8. ROUTING FUNCTION
# =====================================================

def route(state: State):

    return state["next_agent"]


# =====================================================
# 9. BUILD GRAPH
# =====================================================

builder = StateGraph(State)

builder.add_node("router", router_node)
builder.add_node("research_agent", research_node)
builder.add_node("validation_agent", validation_node)
builder.add_node("report_agent", report_node)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    route,
    {
        "research_agent": "research_agent",
        "validation_agent": "validation_agent",
        "report_agent": "report_agent"
    }
)

builder.add_edge("research_agent", "router")
builder.add_edge("validation_agent", "router")
builder.add_edge("report_agent", END)

graph = builder.compile()


# =====================================================
# 10. EXECUTE
# =====================================================

response = graph.invoke(
    {
        "topic": "CVE-2021-44228",
        "research": "",
        "validation": "",
        "report": "",
        "next_agent": ""
    },
    config={"recursion_limit": 30}
)

print("\nFINAL REPORT:")
print(response["report"])


1. START
      |
2. router_node()
      |
      | Returns:
      | {"next_agent": "research_agent"}
      |
3. route(state)
      |
      | Returns "research_agent"
      |
4. LangGraph checks conditional edges:
      |
      | "research_agent": "research_agent"
      |
5. LangGraph executes research_node()
      |
6. research_node() returns research findings
      |
7. LangGraph updates shared state
      |
8. Execution returns to router_node()

builder.add_conditional_edges(
    "router",
    route,
    {
        these are keys are values of route function and value in this key is the node name which get called
        "research_agent": "research_agent",
        "validation_agent": "validation_agent",
        "report_agent": "report_agent"
    }
)



builder.add_conditional_edges(
    "router",  # Source node
    route,     # Function LangGraph will call after source node
    {...}      # Mapping from returned key to destination node
)

This tells LangGraph:
Whenever the "router" node finishes executing, call the route() function with the updated state. Then use its return value to decide which node to execute next.